# กำหนด ROI เสาจากกล้องด้านหน้าแบบหมุนได้

ไฟล์นี้ใช้เชื่อมต่อ Google Drive เลือกภาพจากวิดีโอมุมกล้องด้านหน้า และกำหนดพื้นที่สนใจ (ROI) ของเสา เช่น C01, C02 ด้วยกรอบสี่เหลี่ยมที่ปรับตำแหน่ง ขนาด และมุมหมุนได้ เพื่อนำค่าพิกัดไปใช้ในโค้ดตรวจจับการเทคอนกรีต

รันเซลล์ตามลำดับจากบนลงล่าง โดยระบุพาธวิดีโอและเวลาของภาพที่ต้องการใช้ในขั้นตอนที่ 2


## 1. เชื่อมต่อ Google Drive

เชื่อมต่อ Google Drive เพื่ออ่านวิดีโอ เมื่อระบบขอสิทธิ์ ให้เลือกบัญชีที่เก็บไฟล์วิดีโอ


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. เปิดวิดีโอและเลือกภาพสำหรับกำหนด ROI

ระบุพาธวิดีโอมุมกล้องด้านหน้าใน `video_path` และเวลาของภาพเป็นวินาทีใน `time_sec` ตัวอย่างนี้ใช้ภาพจากวินาทีที่ 10 ของ `front.mp4`

เซลล์นี้จะดึงภาพและแปลงเป็น RGB เก็บในตัวแปร `img` สำหรับเครื่องมือกำหนด ROI ด้านล่าง


In [ ]:
import cv2

video_path = "/content/drive/MyDrive/Video_PRO/front.mp4"
time_sec = 10

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_MSEC, time_sec * 1000)
ret, frame = cap.read()
cap.release()

if not ret:
    raise ValueError("เปิดวิดีโอหรือดึงภาพไม่ได้ ตรวจสอบ video_path และ time_sec")

img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print("ขนาดภาพ =", img.shape)


## 3. นำเข้าไลบรารี

นำเข้าไลบรารีสำหรับวาดกรอบและจัดการพิกัด แสดงภาพ และสร้างตัวเลื่อนกับปุ่มควบคุมใน Notebook


In [11]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

## 4. สร้างฟังก์ชันวาดกรอบหมุน

ฟังก์ชัน `draw_rotated_box()` สร้างกรอบจากจุดศูนย์กลาง `cx, cy` ความกว้าง `w` ความสูง `h` และมุมหมุน `angle` พร้อมแสดงชื่อเสาบนภาพ


In [12]:
def draw_rotated_box(image, cx, cy, w, h, angle, label=None, color=(255, 0, 0), thickness=3):
    vis = image.copy()

    rect = ((cx, cy), (w, h), angle)
    box = cv2.boxPoints(rect)   # ได้ 4 มุม
    box = np.int32(box)

    cv2.polylines(vis, [box], isClosed=True, color=color, thickness=thickness)

    if label is not None:
        x_text = int(box[:, 0].min())
        y_text = int(box[:, 1].min()) - 10
        cv2.putText(
            vis,
            label,
            (x_text, max(20, y_text)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 0),
            2
        )

    return vis, box

## 5. เตรียมตัวเลื่อนและปุ่มควบคุม

อ่านขนาดภาพจาก `img` แล้วสร้างตัวเลือกหมายเลข Column และตัวเลื่อนสำหรับปรับกรอบ

| ตัวควบคุม | ความหมาย |
| --- | --- |
| Column | หมายเลขเสา เช่น 1 หมายถึง C01 |
| cx | ตำแหน่งศูนย์กลางกรอบในแนวนอน (พิกเซล) |
| cy | ตำแหน่งศูนย์กลางกรอบในแนวตั้ง (พิกเซล) |
| w | ความกว้างกรอบ (พิกเซล) |
| h | ความสูงกรอบ (พิกเซล) |
| angle | มุมหมุนกรอบ ตั้งแต่ −90 ถึง 90 องศา |

ค่าที่บันทึกจะเก็บไว้ใน `column_rois` การรันเซลล์นี้ใหม่จะเริ่มตัวแปรดังกล่าวเป็นค่าว่าง


In [13]:
H, W = img.shape[:2]

column_rois = {}

column_input = widgets.BoundedIntText(
    value=1, min=1, max=50, description="Column:"
)

cx_slider = widgets.IntSlider(
    value=W//2, min=0, max=W-1, step=1,
    description="cx:", continuous_update=False
)

cy_slider = widgets.IntSlider(
    value=H//2, min=0, max=H-1, step=1,
    description="cy:", continuous_update=False
)

w_slider = widgets.IntSlider(
    value=120, min=10, max=W, step=1,
    description="w:", continuous_update=False
)

h_slider = widgets.IntSlider(
    value=300, min=10, max=H, step=1,
    description="h:", continuous_update=False
)

angle_slider = widgets.IntSlider(
    value=0, min=-90, max=90, step=1,
    description="angle:", continuous_update=False
)

save_button = widgets.Button(description="Save Column", button_style="success")
show_all_button = widgets.Button(description="Show All", button_style="info")
print_button = widgets.Button(description="Print Dict", button_style="warning")

output = widgets.Output()

## 6. สร้างภาพตัวอย่างระหว่างปรับกรอบ

กำหนดชื่อเสาเป็น C01, C02 ตามหมายเลขที่เลือก และสร้างภาพตัวอย่างของกรอบปัจจุบันกับกรอบที่บันทึกไว้แล้ว

กรอบสีแดงคือกรอบที่กำลังปรับ ส่วนกรอบสีเขียวคือกรอบที่บันทึกไว้แล้ว ด้านล่างภาพจะแสดงค่าพิกัดและมุมหมุนของกรอบปัจจุบัน


In [14]:
def get_column_name():
    return f"C{column_input.value:02d}"


def preview_rotated(change=None):
    name = get_column_name()

    cx = cx_slider.value
    cy = cy_slider.value
    w_box = w_slider.value
    h_box = h_slider.value
    angle = angle_slider.value

    vis = img.copy()

    # วาด ROI ที่บันทึกไว้แล้ว
    for col_name, data in column_rois.items():
        rect = ((data["cx"], data["cy"]), (data["w"], data["h"]), data["angle"])
        box = cv2.boxPoints(rect)
        box = np.int32(box)

        cv2.polylines(vis, [box], True, (0, 255, 0), 2)

        x_text = int(box[:, 0].min())
        y_text = int(box[:, 1].min()) - 5

        cv2.putText(
            vis,
            col_name,
            (x_text, max(20, y_text)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    # วาด ROI ที่กำลังแก้
    vis, box = draw_rotated_box(
        vis, cx, cy, w_box, h_box, angle,
        label=name, color=(255, 0, 0), thickness=3
    )

    with output:
        clear_output(wait=True)

        plt.figure(figsize=(16, 9))
        plt.imshow(vis)
        plt.axis("off")
        plt.show()

        print(f'{name}: {{"cx": {cx}, "cy": {cy}, "w": {w_box}, "h": {h_box}, "angle": {angle}}}')

## 7. เตรียมการบันทึกและแสดงค่าพิกัด

กำหนดการทำงานของปุ่มทั้งสาม

- **Save Column**: เก็บค่ากรอบของเสาที่เลือกใน `column_rois` หากใช้หมายเลขเดิม จะบันทึกทับค่าของเสานั้น
- **Show All**: แสดงกรอบของเสาที่บันทึกไว้ทั้งหมด
- **Print Dict**: แสดง `column_rois` เพื่อคัดลอกไปใช้ในโค้ดหลัก


In [15]:
def save_column(button):
    name = get_column_name()

    column_rois[name] = {
        "cx": cx_slider.value,
        "cy": cy_slider.value,
        "w": w_slider.value,
        "h": h_slider.value,
        "angle": angle_slider.value
    }

    preview_rotated()

    with output:
        print("✅ Saved", name)


def show_all(button):
    vis = img.copy()

    for name, data in column_rois.items():
        rect = ((data["cx"], data["cy"]), (data["w"], data["h"]), data["angle"])
        box = cv2.boxPoints(rect)
        box = np.int32(box)

        cv2.polylines(vis, [box], True, (255, 0, 0), 3)

        x_text = int(box[:, 0].min())
        y_text = int(box[:, 1].min()) - 5

        cv2.putText(
            vis,
            name,
            (x_text, max(20, y_text)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2
        )

    with output:
        clear_output(wait=True)

        plt.figure(figsize=(16, 9))
        plt.imshow(vis)
        plt.axis("off")
        plt.show()


def print_dict(button):
    with output:
        clear_output()
        print("column_rois = {")
        for name, data in sorted(column_rois.items()):
            print(f'    "{name}": {data},')
        print("}")

## 8. เชื่อมตัวเลื่อนและปุ่มกับคำสั่ง

เชื่อมตัวเลื่อนและหมายเลข Column ให้แสดงภาพตัวอย่างเมื่อเปลี่ยนค่า พร้อมเชื่อมปุ่มแต่ละปุ่มกับฟังก์ชันที่กำหนดไว้


In [16]:
for slider in [cx_slider, cy_slider, w_slider, h_slider, angle_slider]:
    slider.observe(preview_rotated, names="value")

column_input.observe(preview_rotated, names="value")

save_button.on_click(save_column)
show_all_button.on_click(show_all)
print_button.on_click(print_dict)

## 9. เปิดเครื่องมือและกำหนด ROI

รันเซลล์ด้านล่างเพื่อแสดงเครื่องมือ แล้วทำตามขั้นตอนดังนี้

1. เลือกหมายเลข **Column** ของเสาที่ต้องการกำหนด
2. ปรับ `cx, cy` เพื่อเลื่อนกรอบให้ตรงพื้นที่เสา
3. ปรับ `w, h` เพื่อกำหนดขนาดกรอบ และปรับ `angle` ให้ตรงแนวเอียงของเสา
4. กด **Save Column** เพื่อบันทึกค่าของเสานั้น
5. เปลี่ยนหมายเลข Column และทำซ้ำจนครบทุกเสา
6. กด **Show All** เพื่อตรวจสอบกรอบทั้งหมด แล้วกด **Print Dict** เพื่อคัดลอกค่าพิกัดไปใช้ในโค้ดหลัก

ค่าของแต่ละเสาอยู่ในรูปแบบ `{"cx": ..., "cy": ..., "w": ..., "h": ..., "angle": ...}`

ปุ่ม **Save Column** เก็บค่าไว้ในตัวแปรระหว่างใช้งาน จึงควรคัดลอกค่าพิกัดก่อนปิดเซสชันหรือเริ่มกำหนด ROI ของมุมกล้องอื่น


In [17]:
display(
    column_input,
    cx_slider,
    cy_slider,
    w_slider,
    h_slider,
    angle_slider,
    widgets.HBox([save_button, show_all_button, print_button]),
    output
)

preview_rotated()

BoundedIntText(value=1, description='Column:', max=50, min=1)

IntSlider(value=640, continuous_update=False, description='cx:', max=1279)

IntSlider(value=360, continuous_update=False, description='cy:', max=719)

IntSlider(value=120, continuous_update=False, description='w:', max=1280, min=10)

IntSlider(value=300, continuous_update=False, description='h:', max=720, min=10)

IntSlider(value=0, continuous_update=False, description='angle:', max=90, min=-90)

Output()